In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time
import threading

counter = 0
lock = threading.Lock()

def thread_function():
  global counter
  for _ in range(100000):
    with lock:
      x = counter
      time.sleep(0)
      counter = x + 1

with ThreadPoolExecutor(4) as executor:
  for _ in range(4):
    executor.submit(thread_function)

print(counter)

400000


In [ ]:
# bounded producer/consumer buffer with threading.Condition

import threading
from collections import deque

bufferLock = threading.Lock()
bufferNonEmpty = threading.Condition(bufferLock)
bufferNonFull = threading.Condition(bufferLock)
items = deque()
capacity = 5

def producer(item):
  with bufferLock:
    # while items is already at capacity, wait for condition to be notified by a consumer thread
    while len(items) == capacity:
      bufferNonFull.wait()
    items.append(item)
    bufferNonEmpty.notify() # notify any waiting consumer threads

def consumer():
  with bufferLock:
    while len(items) == 0:
      bufferNonEmpty.wait()
    ret = items.popleft()
    bufferNonFull.notify()
    return ret

In [ ]:
# bounded producer/consumer buffer with queue.Queue

import queue

items = queue.Queue(maxsize=5)

def producer(item):
  items.put(item)

def consumer():
  return items.get()

In [ ]:
# pure python loop vs numpy matrix multiplication

import time
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from threadpoolctl import threadpool_limits, threadpool_info

A = np.random.rand(500, 500)
B = np.random.rand(500, 500)

def pure_python():
  total = 0
  for i in range(1000000):
    total += i
  return total

def np_matmul():
  return A @ B

def run_threads(max_threads, thread_function, total_threads):
  with ThreadPoolExecutor(max_threads) as executor:
    for i in range(total_threads):
      executor.submit(thread_function)

def time_thread_fn(thread_function, total_threads, repeats=3):
    thread_function()  # warm-up run, not timed
    for num_threads in [1, 2, 4]:
        times = []
        for _ in range(repeats):
            start = time.perf_counter()
            run_threads(num_threads, thread_function, total_threads)
            times.append(time.perf_counter() - start)
        print(f"{num_threads} thread(s): best {min(times):.3f}s")

print("Testing pure Python function...")
time_thread_fn(pure_python, 16)
print()

# BLAS is the C code (rather than Python) that is run to calculate the matmul
with threadpool_limits(limits=1, user_api="blas"): # this line tells BLAS to only use one thread per matmul call
  print("Testing numpy matmul function...")
  time_thread_fn(np_matmul, 128)

# Note that with matmul, there is a speedup from 1 to 2 threads but a slowdown from 2 to 4 threads. This is because the notebook only has 2 CPUs available, so extra threads will just be waiting for a core.


Testing pure Python function...
1 thread(s): best 1.045s
2 thread(s): best 1.079s
4 thread(s): best 1.130s

Testing numpy matmul function...
1 thread(s): best 1.624s
2 thread(s): best 1.163s
4 thread(s): best 1.267s


In [ ]:
pip install sympy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 10.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 18.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [sympy]  WARNING: The script isympy is installed in '/Library/Frameworks/Python.framework/Versions/3.14/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [sympy]

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# counting number of primes [1,8000000]
from sympy import isprime
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import multiprocessing as mp
import time

# prediction: because of the GIL, 8 threads don't provide any speedup from serial execution, and with the overhead of switching threads, it will slightly slower. Meanwhile, 8 processes should be much faster.

def count_primes(start, end): # inclusive
  count = 0
  for i in range(start, end+1):
    if isprime(i):
      count += 1
  return count

# serial execution
print("Serial execution result:")
start = time.perf_counter()
print(count_primes(1,8000000))
execution_time = time.perf_counter() - start
print(f"Execution time: {execution_time}")
print()

# 8 threads
print("8 threads result:")
start = time.perf_counter()
start_vals = [1+1000000*i for i in range(8)]
end_vals = [1000000*(i+1) for i in range(8)]
with ThreadPoolExecutor(8) as executor:
  counts = executor.map(count_primes, start_vals, end_vals)
print(sum(counts))
execution_time = time.perf_counter() - start
print(f"Execution time: {execution_time}")
print()

# 8 processes
print("8 processes result:")
start = time.perf_counter()
start_vals = [1+1000000*i for i in range(8)]
end_vals = [1000000*(i+1) for i in range(8)]
with ProcessPoolExecutor(8, mp_context=mp.get_context("fork")) as executor:
  counts = executor.map(count_primes, start_vals, end_vals)
print(sum(counts))
execution_time = time.perf_counter() - start
print(f"Execution time: {execution_time}")


Serial execution result:
539777
Execution time: 2.681460125138983

8 threads result:
539777
Execution time: 2.6425279590766877

8 processes result:
539777
Execution time: 0.43212154088541865


In [7]:
pip install requests

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [requests]

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
# fetching 200 urls
import requests
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import time
import multiprocessing as mp

# prediction: because the CPU is idle for much of the time spent fetching requests, incorporating threads will be able to speedup execution compared to serial execution. Using processes instead will likely also be faster than serial execution but may be slightly slower than using threading because of the overhead of switching processes.

urls = [f"https://jsonplaceholder.typicode.com/comments/{i}" for i in range(1, 241)]

def fetch_urls(urls):
  for url in urls:
    response = requests.get(url, timeout=10)

# serial execution
print("Serial execution result:")
start = time.perf_counter()
fetch_urls(urls)
execution_time = time.perf_counter() - start
print(f"Execution time: {execution_time}")
print()

# 8 threads
print("8 threads result:")
start = time.perf_counter()
urls_per_thread = [urls[i::8] for i in range(8)]
with ThreadPoolExecutor(8) as executor:
  counts = executor.map(fetch_urls, urls_per_thread)
execution_time = time.perf_counter() - start
print(f"Execution time: {execution_time}")
print()

# 8 processes
print("8 processes result:")
start = time.perf_counter()
urls_per_process = [urls[i::8] for i in range(8)]
with ProcessPoolExecutor(8, mp_context=mp.get_context("fork")) as executor:
  counts = executor.map(fetch_urls, urls_per_process)
execution_time = time.perf_counter() - start
print(f"Execution time: {execution_time}")


Serial execution result:
Execution time: 66.59987912513316

8 threads result:
Execution time: 5.92957124998793

8 processes result:
Execution time: 7.455307292053476
